# 02 Can neighbours predict the missing hours?
Hide the hours of an area we do know, copy them from the nearest known area, and measure.

In [ ]:
import geopandas as gpd, pandas as pd, matplotlib.pyplot as plt

areas = gpd.read_parquet("../data/processed/parking_rules.parquet")  # 8,754 areas, metric CRS
print(len(areas), "areas |", areas.crs.to_epsg())

In [ ]:
known = areas[(areas["status"] == "official") & areas["hours"].notna()].reset_index(drop=True)
missing = areas[areas["status"] == "missing_hours"]
print(len(known), "with hours |", len(missing), "missing")

## Accuracy of copying a neighbour

In [ ]:
def nearest(row, pool, same_class):
    cand = pool[pool.index != row.name]
    if same_class:
        cand = cand[cand["luokka"] == row["luokka"]]
    if cand.empty:
        return None, None
    d = cand.distance(row.geometry)
    return d.min(), cand.loc[d.idxmin(), "hours"]

sample = known.sample(500, random_state=0)
rows = [{"same_class": s, "dist": (r_ := nearest(r, known, s))[0], "match": r_[1] == r["hours"]}
        for _, r in sample.iterrows() for s in (False, True)]
res = pd.DataFrame(rows).dropna()
res.groupby("same_class")["match"].mean()

Accuracy falls with distance. This is the basis for the confidence threshold.

In [ ]:
res["dist_bin"] = pd.cut(res["dist"], [0, 10, 50, 200, 1e9])
res.groupby(["same_class", "dist_bin"], observed=True)["match"].agg(["mean", "size"])

## The catch
Areas with missing hours sit far from areas with known hours, so the test above is optimistic.

In [ ]:
d = pd.Series([known[known["luokka"] == r["luokka"]].distance(r.geometry).min()
               for _, r in missing.iterrows()])
print(d.describe())
print("within 50 m:", f"{(d <= 50).mean():.1%}")